In [2]:
from langgraph.graph import MessagesState
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, RemoveMessage
from langgraph.graph import StateGraph, START
from langgraph.checkpoint.memory import InMemorySaver

In [3]:
load_dotenv()

True

In [4]:
llm = ChatGroq(model="openai/gpt-oss-120b")

In [5]:
class ChatState(MessagesState):
    summary:str

In [6]:
def summarize_convo(state:ChatState):

    existing_summary =  state['summary']

    if existing_summary:
        prompt=(
            f"Existing summary {existing_summary} \n\n "
            "Extend the summary using the new conversation above"
        )
    else :  
        prompt = "Summarize the conversation above"

    messages_for_summary = state['summary'] + [HumanMessage(content=prompt)]

    response = llm.invoke(messages_for_summary)

    messages_to_delete = state['messages'][:-2]

    return {
        "summary":response.content,
        "messages":[RemoveMessage(id=m.id) for m in messages_to_delete]
    }

In [8]:
def chat(state:ChatState):
    msg=[]

    if state['summary']:
        msg.append({
            "role":"system",
            "content":f"Conversation summary:\n{state['summary']}"
            })
        
    msg.extend(state["messages"])

    print(msg)

    response = llm.invoke(msg)

    return {"messages":[response]}
    

In [9]:
def should_summarize(state:ChatState):
    return len(state["messages"])>6


In [10]:
builder = StateGraph(ChatState)

builder.add_node("chat", chat)
builder.add_node("summarize", summarize_convo)

builder.add_edge(START, "chat")

builder.add_conditional_edges(
    "chat",
    should_summarize,
    {
        True: "summarize",
        False: "__end__",
    }
)

builder.add_edge("summarize", "__end__")

In [11]:
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)